# 02 — Modélisation : tester des modèles et choisir le meilleur

**Démarche**
1. Préparer les données avec le pipeline défini après l'EDA.
2. Comparer 6 modèles en validation croisée stratifiée (5 plis) sur le **train**.
3. Optimiser les hyperparamètres des candidats.
4. Choisir le modèle selon 4 critères : performance, stabilité, temps, interprétabilité.
5. Ajuster le seuil de décision.
6. Évaluer **une seule fois** sur le **test**, interpréter et analyser l'équité.

**Métrique principale : ROC-AUC.** Elle mesure la capacité du modèle à classer un individu >50K
au-dessus d'un individu <=50K, indépendamment du seuil et du déséquilibre des classes.
On regarde aussi F1 et PR-AUC, plus centrées sur la classe minoritaire.

In [ ]:
import sys
from pathlib import Path

# Permet d'importer le package src depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import FIGURES_DIR

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 30)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    """Enregistre la figure dans reports/figures/ pour la présentation."""
    fig.savefig(FIGURES_DIR / name, dpi=120, bbox_inches="tight")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from src.config import RANDOM_STATE, SENSITIVE_FEATURES
from src.data.load_data import load_train_test, split_X_y
from src.models.evaluate import (best_threshold, compute_metrics, cross_validate_models,
                                   fairness_report, permutation_importance_df, plot_confusion,
                                   plot_cv_comparison, plot_importance, plot_pr_curves,
                                   plot_roc_curves)
from src.features.build_features import build_preprocessor
from src.models.models import PARAM_DISTRIBUTIONS, get_pipelines, save_model, tune

# FAST = True pour tester rapidement que tout tourne ; False pour les vrais résultats.
FAST = False
N_ITER, CV = (3, 3) if FAST else (20, 5)

train, test = load_train_test()
X_train, y_train = split_X_y(train)
X_test, y_test = split_X_y(test)
print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"Taux de >50K — train : {y_train.mean():.1%} | test : {y_test.mean():.1%}")

## 1. Le preprocessing

Le preprocessing est un `ColumnTransformer` scikit-learn placé **dans** chaque pipeline de modèle :
il est réappris sur les plis d'entraînement à chaque itération de la validation croisée,
ce qui évite toute fuite d'information du jeu de validation.

In [ ]:
prep = build_preprocessor().fit(X_train)
names = prep.get_feature_names_out()
print(f"{X_train.shape[1]} colonnes brutes -> {len(names)} variables après preprocessing")
pd.DataFrame(prep.transform(X_train.head()), columns=names).round(2)

## 2. Comparaison des modèles (hyperparamètres par défaut)

| Modèle | Pourquoi le tester |
|---|---|
| Baseline (classe majoritaire) | Le score minimum à battre |
| Régression logistique | Modèle linéaire simple, rapide, coefficients interprétables |
| Arbre de décision | Non linéaire et lisible, mais instable / surapprend facilement |
| KNN | Approche par similarité, sans apprentissage de paramètres |
| Random Forest | Moyenne de nombreux arbres : réduit la variance |
| Gradient Boosting | Arbres construits pour corriger les erreurs des précédents : souvent le meilleur sur données tabulaires |

In [ ]:
cv_default = cross_validate_models(get_pipelines(), X_train, y_train, cv=CV)
cols = ["modele", "roc_auc_mean", "roc_auc_std", "f1_mean", "pr_auc_mean", "accuracy_mean", "fit_time_s"]
cv_default[cols].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_cv_comparison(cv_default, ax=ax)
save(fig, "model_cv_comparison.png")

**Lecture** (vérifier sur vos sorties)
- La baseline a une accuracy d'environ 0,76 mais un F1 nul et une ROC-AUC de 0,5 :
  elle confirme que l'accuracy seule ne suffit pas.
- Tous les modèles battent nettement la baseline.
- L'arbre de décision seul est le moins bon des vrais modèles et le plus variable.
- Les méthodes d'ensemble (Random Forest, Gradient Boosting) et la régression logistique
  sont en tête ; KNN est correct mais lent à la prédiction.

## 3. Optimisation des hyperparamètres

`RandomizedSearchCV` : on tire au hasard `N_ITER` combinaisons dans un espace de recherche
et on garde celle qui maximise la ROC-AUC en validation croisée.
Plus efficace qu'une grille complète pour un temps de calcul limité.

In [ ]:
searches, rows = {}, []
for name in PARAM_DISTRIBUTIONS:
    search = tune(name, X_train, y_train, n_iter=N_ITER, cv=CV)
    searches[name] = search
    i = search.best_index_
    rows.append({"modele": name,
                 "roc_auc_defaut": cv_default.set_index("modele").loc[name, "roc_auc_mean"],
                 "roc_auc_tune": search.best_score_,
                 "roc_auc_std": search.cv_results_["std_test_score"][i],
                 "fit_time_s": search.cv_results_["mean_fit_time"][i]})
    print(f"{name:25s} -> {search.best_params_}")

cv_tuned = pd.DataFrame(rows).sort_values("roc_auc_tune", ascending=False).reset_index(drop=True)
cv_tuned["gain"] = cv_tuned["roc_auc_tune"] - cv_tuned["roc_auc_defaut"]
cv_tuned.round(4)

## 4. Choix du modèle

On choisit selon **4 critères** :
1. **Performance** : ROC-AUC moyenne en validation croisée (critère principal).
2. **Stabilité** : écart-type faible entre les plis.
3. **Coût** : temps d'entraînement et de prédiction.
4. **Interprétabilité** : peut-on expliquer les décisions ?

In [ ]:
best_name = cv_tuned.loc[0, "modele"]
best_model = searches[best_name].best_estimator_
print(f"Modèle retenu : {best_name}")
print(f"ROC-AUC CV = {cv_tuned.loc[0, 'roc_auc_tune']:.4f} ± {cv_tuned.loc[0, 'roc_auc_std']:.4f}")
print("Hyperparamètres :", searches[best_name].best_params_)

## 5. Ajustement du seuil de décision

Par défaut, un modèle prédit `>50K` si la probabilité dépasse 0,5. Avec des classes
déséquilibrées, ce seuil n'est pas forcément le meilleur. On cherche le seuil qui
maximise le F1, **à partir de prédictions hors échantillon sur le train**
(`cross_val_predict`) : le test reste intact.

In [ ]:
oof = cross_val_predict(best_model, X_train, y_train, method="predict_proba", n_jobs=-1,
                        cv=StratifiedKFold(CV, shuffle=True, random_state=RANDOM_STATE))[:, 1]
threshold, f1_oof = best_threshold(y_train, oof)
print(f"Seuil optimal : {threshold:.3f} (F1 hors échantillon : {f1_oof:.4f})")
print(f"F1 avec le seuil 0,5 : {compute_metrics(y_train, (oof >= 0.5).astype(int), oof)['f1']:.4f}")

## 6. Évaluation finale sur le jeu de test

C'est la **seule** fois où le jeu de test est utilisé. Ses scores estiment la performance
du modèle sur des données nouvelles.

In [ ]:
test_scores = {n: s.best_estimator_.predict_proba(X_test)[:, 1] for n, s in searches.items()}
y_score = test_scores[best_name]
y_pred = (y_score >= threshold).astype(int)

all_models = pd.DataFrame({n: compute_metrics(y_test, (s >= 0.5).astype(int), s)
                           for n, s in test_scores.items()}).T
print("Tous les modèles tunés, seuil 0,5 :")
display(all_models.round(4))

final = pd.DataFrame({
    "seuil 0,5": compute_metrics(y_test, (y_score >= 0.5).astype(int), y_score),
    f"seuil {threshold:.2f}": compute_metrics(y_test, y_pred, y_score),
}).T
print(f"\n{best_name} :")
final.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
plot_roc_curves(test_scores, y_test, ax=axes[0])
plot_pr_curves(test_scores, y_test, ax=axes[1])
plot_confusion(y_test, y_pred, title=f"{best_name} (seuil {threshold:.2f})", ax=axes[2])
fig.tight_layout()
save(fig, "model_test_curves.png")

**Lecture**
- Les scores de test sont proches de ceux de la validation croisée : **pas de surapprentissage**.
- Abaisser le seuil augmente le recall (on détecte plus de >50K) au prix d'un peu de precision :
  c'est un choix métier (que coûte un faux positif par rapport à un faux négatif ?).

## 7. Interprétation : quelles variables comptent ?

**Importance par permutation** : on mélange une variable au hasard sur le jeu de test et on
mesure la perte de ROC-AUC. Méthode valable pour n'importe quel modèle, sur les variables d'origine.

In [ ]:
importances = permutation_importance_df(best_model, X_test, y_test, n_repeats=2 if FAST else 5)
fig, ax = plt.subplots(figsize=(8, 5))
plot_importance(importances, ax=ax)
save(fig, "model_permutation_importance.png")
importances.round(4)

Comparaison avec un modèle directement interprétable : les coefficients de la régression logistique.

In [ ]:
logreg = searches["Régression logistique"].best_estimator_
coefs = pd.Series(logreg.named_steps["model"].coef_[0],
                  index=logreg.named_steps["prep"].get_feature_names_out())
top = pd.concat([coefs.nlargest(8), coefs.nsmallest(8)]).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.values, color=np.where(top.values > 0, "#55A868", "#C44E52"))
ax.set(title="Régression logistique : coefficients les plus forts",
       xlabel="Coefficient (> 0 : augmente la probabilité de >50K)")
save(fig, "model_logreg_coefs.png")

## 8. Analyse d'équité (fairness)

Les données de 1994 contiennent de fortes inégalités de revenus selon le sexe et l'origine.
Un modèle entraîné sur ces données risque de les **reproduire**. On compare, par groupe :
- `taux_predit_50k` : part des personnes prédites >50K ;
- `tpr` (recall) : parmi les vrais >50K, part détectée ;
- `fpr` : parmi les vrais <=50K, part classée à tort >50K.

In [ ]:
for col in SENSITIVE_FEATURES:
    print(f"--- {col} ---")
    display(fairness_report(y_test, y_pred, X_test[col]).round(3))

**Lecture** (vérifier sur vos sorties) : si `tpr` et `fpr` diffèrent fortement entre hommes et
femmes, le modèle ne commet pas les mêmes erreurs selon le groupe. Retirer simplement la variable
`sex` ne suffit pas, car `relationship` (Husband / Wife) la contient indirectement.
Pistes : rééquilibrage, seuils par groupe, bibliothèque `fairlearn`.
Ce point est à mentionner dans la présentation.

## 9. Conclusion : quel modèle et pourquoi ?

> **À compléter avec vos chiffres** une fois le notebook exécuté (les tableaux ci-dessus).

Trame de justification :
- **Modèle retenu : …** (ROC-AUC CV = … ± …, ROC-AUC test = …, F1 test = … au seuil …).
- **Performance** : meilleur score sur la métrique principale, et sur F1 / PR-AUC.
- **Stabilité** : écart-type faible entre les plis, scores de test proches de la CV.
- **Coût** : temps d'entraînement raisonnable (… s).
- **Interprétabilité** : moins direct que la régression logistique, mais l'importance par
  permutation montre des variables cohérentes avec l'EDA (statut marital / relationship,
  gains en capital, éducation, âge).
- **Alternative** : la régression logistique, un peu moins performante (… points de ROC-AUC),
  mais entièrement explicable. Le choix dépend du besoin : précision maximale ou transparence.
- **Limites** : données de 1994, biais selon le sexe et l'origine, seuil à adapter au besoin métier.

In [ ]:
import json

from src.config import MODEL_META_PATH, MODEL_PATH

# Même format que `python -m src.models.train` : c'est ce fichier que charge l'API.
save_model(best_model, MODEL_PATH)
meta = {"modele": best_name, "hyperparametres": searches[best_name].best_params_,
        "seuil": threshold, "test": final.round(4).to_dict(orient="index")}
MODEL_META_PATH.write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"Modèle sauvegardé : {MODEL_PATH}")